# ragfallback — silent RAG failures, caught on real data

[![PyPI](https://img.shields.io/pypi/v/ragfallback)](https://pypi.org/project/ragfallback/)  [![GitHub](https://img.shields.io/badge/GitHub-ragfallback-black?logo=github)](https://github.com/irfanalidv/ragfallback)

This notebook walks through the full ragfallback pipeline on 50 real Wikipedia passages from SQuAD (CC BY-SA 4.0). No API keys, no paid services — just run all cells top to bottom.

What gets covered:
- **Ingest** — chunk quality, embedding validation, metadata sanitization
- **Index** — retrieval health check, stale index detection
- **Retrieval** — hybrid BM25 fallback, automatic failover
- **Generation** — context window budget, chunk boundary stitching
- **Evaluation** — recall, faithfulness, overall score with no LLM judge

In [ ]:
# install — takes about a minute, dependency warnings are fine to ignore
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ragfallback[chroma,huggingface,real-data,hybrid]"],
    check=True
)
print("done")

In [ ]:
# silence tokenizer and huggingface noise
import warnings, logging, os
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset
from langchain_core.documents import Document

ds = load_dataset("rajpurkar/squad", split="validation")

seen, docs, probes = set(), [], []
for row in ds:
    ctx = row["context"].strip()
    if ctx not in seen and len(seen) < 50:
        seen.add(ctx)
        docs.append(Document(page_content=ctx, metadata={"source": "squad"}))
    if row["answers"]["text"] and len(probes) < 20:
        probes.append({
            "question": row["question"],
            "ground_truth": row["answers"]["text"][0]
        })

print(f"{len(docs)} passages, {len(probes)} Q&A pairs loaded")
print(f"\nexample passage: {docs[0].page_content[:180]}...")
print(f"\nexample Q:  {probes[0]['question']}")
print(f"example A:  {probes[0]['ground_truth']}")

## Ingest pipeline

Three things that should happen before any document reaches a vector store — chunk quality check, embedding dimension validation, and metadata sanitization. Skip any of these and you get silent failures downstream.

In [ ]:
from ragfallback.diagnostics import ChunkQualityChecker, EmbeddingGuard, sanitize_documents

# chunk quality — catches too-short, too-long, mid-sentence, duplicate chunks
report = ChunkQualityChecker(min_chars=100, max_chars=8000).check(docs)
print(report.summary())

if report.has_issues:
    fixed_texts = ChunkQualityChecker().auto_fix(docs)
    docs = [Document(page_content=t, metadata={"source": "squad"}) for t in fixed_texts]
    print(f"auto-fixed down to {len(docs)} clean chunks")

In [ ]:
# embedding guard — validates dimensions before anything is written to the index
from ragfallback.utils.embedding_factory import create_open_source_embeddings

embeddings = create_open_source_embeddings(model_name="all-MiniLM-L6-v2")

result = EmbeddingGuard(expected_dim=384).validate(embeddings)
if result.ok:
    print("embedding dim 384 confirmed")
else:
    print(f"guard failed: {result.message}")

In [ ]:
# metadata sanitizer — Chroma, Pinecone, Qdrant all reject list/dict metadata
# this converts everything to JSON-safe scalars before the write

dirty = Document(
    page_content="a passage with problematic metadata",
    metadata={"tags": ["rag", "test"], "scores": {"relevance": 0.9}, "source": "demo"}
)
clean = sanitize_documents([dirty])

print(f"before: {dirty.metadata}")
print(f"after:  {clean[0].metadata}")

# sanitize the real docs too
docs = sanitize_documents(docs)
print(f"\n{len(docs)} passages ready for indexing")

## Index health

Build the index, then immediately verify it can actually retrieve correct answers. Most people skip this and only find out the index is broken when users complain.

In [ ]:
from langchain_community.vectorstores import Chroma
from ragfallback.diagnostics import RetrievalHealthCheck

store = Chroma.from_documents(docs, embeddings)

# run 10 real Q&A probes against the index
probe_dict = {p["question"]: p["ground_truth"][:60] for p in probes[:10]}
health = RetrievalHealthCheck(k=4).run_substring_probes(store, probe_dict)

print(f"hit rate:    {health.hit_rate:.0%}")
print(f"avg latency: {health.avg_latency_ms:.0f} ms")

if health.hit_rate < 0.8:
    print("\nhit rate is low — worth checking chunk quality or embedding model before serving")

In [ ]:
# stale index detection — SHA256 manifest that catches when source files
# have changed since the last index build
import tempfile, pathlib
from ragfallback.diagnostics import StaleIndexDetector

with tempfile.TemporaryDirectory() as tmpdir:
    doc_path = pathlib.Path(tmpdir) / "policy.md"
    doc_path.write_text("Refund policy: 30 days full refund.")
    manifest_path = pathlib.Path(tmpdir) / "manifest.json"

    det = StaleIndexDetector(manifest_path=str(manifest_path))
    det.record_paths([str(doc_path)])   # called right after index build

    r = det.check_paths([str(doc_path)])
    print(f"right after build — stale: {r.has_stale}")   # False

    # simulate someone editing the source file
    doc_path.write_text("Refund policy updated: 14 days only.")

    r = det.check_paths([str(doc_path)])
    print(f"after file change  — stale: {r.has_stale}")   # True
    print(r.summary())

## Retrieval

Two retrievers for the cases where a single dense retriever isn't enough — hybrid BM25 fallback for keyword-heavy queries, and automatic failover when the primary is down.

In [ ]:
from ragfallback.retrieval import SmartThresholdHybridRetriever, FailoverRetriever

# hybrid: threshold on dense scores, BM25 fills in when nothing passes
hybrid = SmartThresholdHybridRetriever.from_documents(
    docs, store, max_distance=0.5, final_k=4, fetch_k=20
)
q = probes[0]["question"]
results = hybrid.invoke(q)
print(f"query:   {q}")
print(f"chunks:  {len(results)}")

In [ ]:
# failover — primary goes down, secondary picks up transparently
class BrokenRetriever:
    def invoke(self, query, **kwargs):
        raise ConnectionError("primary vector store unreachable")

failover = FailoverRetriever(
    primary=BrokenRetriever(),
    fallback=store.as_retriever(search_kwargs={"k": 4}),
    min_results=1
)

results = failover.invoke(q)
print(f"primary raised ConnectionError, fallback returned {len(results)} chunks")
print(f"first result: {results[0].page_content[:120]}...")

## Generation — context preparation

Before handing chunks to the LLM: trim to fit the token budget, and merge adjacent chunks so answers that span a boundary don't get lost.

In [ ]:
from ragfallback.diagnostics import ContextWindowGuard, OverlappingContextStitcher

retrieved = store.as_retriever(search_kwargs={"k": 6}).invoke(probes[1]["question"])

# ranks chunks by relevance and trims to fit the model's token budget
guard = ContextWindowGuard.from_model_name("gpt-4o")   # 128k token budget
selected, report = guard.select(probes[1]["question"], retrieved, embeddings)

print(f"retrieved {len(retrieved)} chunks, {len(selected)} fit within token budget (trim only kicks in when chunks exceed model limit)")


In [ ]:
# if two adjacent chunks from the same source both got retrieved,
# merge them so the answer at the boundary isn't split

passage = docs[0].page_content
mid = len(passage) // 2
chunk_a = Document(page_content=passage[:mid + 50], metadata={"source": "squad", "chunk_id": 0})
chunk_b = Document(page_content=passage[mid - 50:], metadata={"source": "squad", "chunk_id": 1})

merged = OverlappingContextStitcher().stitch([chunk_a, chunk_b])
print(f"2 adjacent chunks merged into {len(merged)}")
print(f"merged: {len(merged[0].page_content)} chars  vs  {len(chunk_a.page_content)} + {len(chunk_b.page_content)} separate")

## Evaluation

Score the pipeline on real Q&A pairs — recall@k, faithfulness, overall. No LLM judge, runs completely locally.

In [ ]:
from ragfallback.evaluation import RAGEvaluator

retriever = store.as_retriever(search_kwargs={"k": 4})
ev = RAGEvaluator()
scores = []

for probe in probes[:10]:
    retrieved_docs = retriever.invoke(probe["question"])
    answer = retrieved_docs[0].page_content if retrieved_docs else "not found"
    score = ev.evaluate(
        probe["question"],
        answer,
        [d.page_content for d in retrieved_docs],
        ground_truth=probe["ground_truth"],
    )
    scores.append(score)

print(ev.batch_summary(scores))